In [1]:
import json
import os
import pandas as pd
import matplotlib.pyplot as plt

import warnings

In [2]:
warnings.filterwarnings('ignore')

In [3]:
storm_outages = pd.read_parquet('../data/interim/storm_outages_2014_2023.parquet')

In [4]:
# Corroborar que solo haya dos valores
storm_outages.groupby('episode_fips_id').storm_caused_outage.mean().value_counts()

storm_caused_outage
0.0    388113
1.0     24346
Name: count, dtype: int64

In [5]:
episode_fips_id_value_counts = storm_outages.episode_fips_id.value_counts()
episode_fips_id_value_counts_gt1 = episode_fips_id_value_counts[episode_fips_id_value_counts > 1]

In [6]:
storm_outages[storm_outages.episode_fips_id.isin(episode_fips_id_value_counts_gt1.index)].storm_caused_outage.unique()
# Una misma tormeta puede causar varios outages en el mismo county los cuales no están relacionados entre sí. 
storm_outages[storm_outages.episode_fips_id=='107733_06037'].head()

,EPISODE_ID,fips_code_id,episode_description,begin_datetime,end_datetime,storm_duration,episode_fips_id,storm_caused_outage,outage_index_id,outage_start_minus_storm_start,outage_end_minus_storm_end,outage_start_minus_storm_end,outage_end_minus_storm_start,outage_duration,run_start_time_min,run_start_time_max
92952,107733,06037,Due to very dry vegetation and continued hot a...,2016-07-09 11:04:00,2016-07-31 23:59:00,540.916667,107733_06037,1.0,06037__0227,1.007639,-21.124306,-21.530556,1.413889,0.406250,2016-07-10 11:15:00,2016-07-10 21:00:00
92953,107733,06037,Due to very dry vegetation and continued hot a...,2016-07-09 11:04:00,2016-07-31 23:59:00,540.916667,107733_06037,1.0,06037__0228,3.413889,-19.113889,-19.124306,3.424306,0.010417,2016-07-12 21:00:00,2016-07-12 21:15:00
92954,107733,06037,Due to very dry vegetation and continued hot a...,2016-07-09 11:04:00,2016-07-31 23:59:00,540.916667,107733_06037,1.0,06037__0229,4.257639,-18.270139,-18.280556,4.268056,0.010417,2016-07-13 17:15:00,2016-07-13 17:30:00
92955,107733,06037,Due to very dry vegetation and continued hot a...,2016-07-09 11:04:00,2016-07-31 23:59:00,540.916667,107733_06037,1.0,06037__0230,6.278472,-15.457639,-16.259722,7.080556,0.802083,2016-07-15 17:45:00,2016-07-16 13:00:00
92956,107733,06037,Due to very dry vegetation and continued hot a...,2016-07-09 11:04:00,2016-07-31 23:59:00,540.916667,107733_06037,1.0,06037__0231,8.382639,-13.790972,-14.155556,8.747222,0.364583,2016-07-17 20:15:00,2016-07-18 05:00:00


In [7]:
# Simplification, we weill keep the smallest outage_date, and the longest outage_date (all the sepparated outage will be considered as just one big outage.)
storm_outages_g = storm_outages.groupby('episode_fips_id').agg(
    EPISODE_ID=('EPISODE_ID','first'),
    fips_code_id=('fips_code_id','first'),
    episode_description=('episode_description','first'),
    begin_datetime=('begin_datetime','first'),
    end_datetime=('end_datetime','first'),
    storm_duration=('storm_duration','first'),
    #episode_fips_id=('episode_fips_id','first'),
    storm_caused_outage=('storm_caused_outage','max'),
    outage_index_id=('outage_index_id','first'),
    outage_duration =('outage_duration', 'sum'),
    run_start_time_min=('run_start_time_min','min'),
    run_start_time_max=('run_start_time_max','max'),
).reset_index()


In [8]:
storm_outages_g

,episode_fips_id,EPISODE_ID,fips_code_id,episode_description,begin_datetime,end_datetime,storm_duration,storm_caused_outage,outage_index_id,outage_duration,run_start_time_min,run_start_time_max
0,100001_51139,100001,51139,High pressure sprawled over the Mid-Atlantic r...,2015-09-22 23:35:00,2015-09-23 08:00:00,8.416667,0.0,None,0.0,NaT,NaT
1,100001_51165,100001,51165,High pressure sprawled over the Mid-Atlantic r...,2015-09-22 23:35:00,2015-09-23 08:00:00,8.416667,0.0,None,0.0,NaT,NaT
2,100001_51171,100001,51171,High pressure sprawled over the Mid-Atlantic r...,2015-09-22 23:35:00,2015-09-23 08:00:00,8.416667,0.0,None,0.0,NaT,NaT
3,100001_51187,100001,51187,High pressure sprawled over the Mid-Atlantic r...,2015-09-22 23:35:00,2015-09-23 08:00:00,8.416667,0.0,None,0.0,NaT,NaT
4,100002_54023,100002,54023,High pressure sprawled over the Mid-Atlantic r...,2015-09-24 04:30:00,2015-09-24 09:35:00,5.083333,0.0,None,0.0,NaT,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...
412454,99997_88538,99997,88538,Scattered showers and isolated thunderstorms f...,2015-08-11 14:12:00,2015-08-11 17:30:00,3.300000,0.0,None,0.0,NaT,NaT
412455,99998_88536,99998,88536,Strong moisture advection over the Mid-Atlanti...,2015-08-20 19:34:00,2015-08-20 21:36:00,2.033333,0.0,None,0.0,NaT,NaT
412456,99998_88537,99998,88537,Strong moisture advection over the Mid-Atlanti...,2015-08-20 19:34:00,2015-08-20 21:36:00,2.033333,0.0,None,0.0,NaT,NaT
412457,99999_88531,99999,88531,"A cold front moved through the Mid-Atlantic, t...",2015-08-24 18:49:00,2015-08-24 19:51:00,1.033333,0.0,None,0.0,NaT,NaT


In [9]:
meteorological_path = '../data/raw/meteorological'
meteorological_files = os.listdir(meteorological_path)

In [10]:
len(meteorological_files)

20569

In [11]:
def get_data_with_response_variable(met_file):
    try:
    #if True:
        episode_fips_id = met_file.split('.')[0]
        #  Read data
        path = os.path.join(meteorological_path, met_file)
        
        with open(path, 'r') as f:
            data_raw = json.load(f)
        # Make a dataframe from information
        coord0, coord1, coord2 = data_raw.get('geometry').get('coordinates')
        data = pd.DataFrame(data_raw.get('properties').get('parameter'))
        data.index.rename('time', inplace=True)
        data.reset_index(inplace=True)
        data['episode_fips_id'] = episode_fips_id
        data['meteorological_current_datetime_val'] = pd.to_datetime(
            data['time'].apply(lambda x: x[0:4]) + 
            '-' + 
            data['time'].apply(lambda x: x[4:6]) +
            '-' + 
            data['time'].apply(lambda x: x[6:8]) +
            ' ' +
            data['time'].apply(lambda x: x[8:10]) +
            ':00:00'
        )

        data['day_of_year'] = data.meteorological_current_datetime_val.dt.day_of_year
        data['hour_of_day'] = data.meteorological_current_datetime_val.dt.hour
        data['day_of_week'] = data.meteorological_current_datetime_val.dt.day_of_week
        data['month_of_year'] = data.meteorological_current_datetime_val.dt.month
        data['coord0'] = coord0
        data['coord1'] = coord1
        data['coord2'] = coord2
        
        # Modification to change the hour treshold.
        data['meteorological_nextHour_datetime_val'] = data['meteorological_current_datetime_val'] + pd.Timedelta(value=1, unit='hours')
        # Get the response var dataframe
        storm_outage_episode = storm_outages_g[storm_outages_g.episode_fips_id == episode_fips_id]
        meaning_dict = {
            'begin_datetime': 'storm_start', 
            'end_datetime': 'storm_end', 
            'run_start_time_min': 'outage_start', 
            'run_start_time_max': 'outage_end'
        }
        storm_outage_episode.rename(columns=meaning_dict, inplace=True)
        storm_outage_episode_columns = [
            'episode_fips_id', 
            'storm_start', 
            #'storm_end',
            'outage_start',
            'outage_end'
        ]
        # Join information
        data_rv = data.merge(
            storm_outage_episode[storm_outage_episode_columns],
            on='episode_fips_id',
            how='left'
        )
        #data_rv['outage_start_rounded'] = data_rv.outage_start.dt.round('H')
        #data_rv['outage_end_rounded'] = data_rv.outage_end.dt.round('H')
        
        # Buscamos informacion meteorologica antes del inicio del outage, después ya no nos sirve. 
        data_rv = data_rv[data_rv.meteorological_current_datetime_val <= data_rv.outage_start]
        #data_rv = data_rv[data_rv.meteorological_nextHour_datetime_val <= data_rv.outage_start_rounded]
        
        # Queremos 3 horas de info antes del inicio de la tormenta. 
        data_rv = data_rv[data_rv.meteorological_current_datetime_val >= data_rv.storm_start - pd.Timedelta(value=3, unit='hours')]

        data_rv['hours_to_outage'] = (data_rv.outage_start - data_rv.meteorological_current_datetime_val).dt.total_seconds()/3600
        data_rv['outage_in_an_hour'] = (data_rv['hours_to_outage']<=1).astype(int)
        
        #data_rv['test_val'] = data_rv.outage_start - data_rv.meteorological_current_datetime_val
        #data_rv.loc[data_rv.meteorological_nextHour_datetime_val == data_rv.outage_start_rounded, 'outage_in_an_hour'] = 1
        #data_rv.loc[data_rv.meteorological_nextHour_datetime_val != data_rv.outage_start_rounded, 'outage_in_an_hour'] = 0
        data_rv['episode_fips_time_id'] = data_rv.episode_fips_id + '_' + data_rv.time.astype(str)
        data_rv.set_index('episode_fips_time_id', inplace=True)
        return data_rv
    except:
        return None


In [12]:
#get_data_with_response_variable(met_file)

In [13]:
%%time
data_rvs = []
for met_file in meteorological_files:
    data_rvs.append(get_data_with_response_variable(met_file))

CPU times: user 13min 28s, sys: 7.04 s, total: 13min 35s
Wall time: 19min 24s


In [14]:
len(data_rvs)

20569

In [15]:
cleaned_data_rvs = [x for x in data_rvs if x is not None]

In [16]:
all_data = pd.concat(cleaned_data_rvs)

In [17]:
#plt.hist(testing_prueba.time, bins=100)

In [18]:
all_data.outage_in_an_hour.sum()

np.int64(25625)

In [19]:
#all_data.groupby('episode_fips_id').outage_in_an_hour.sum().sort_values()

In [20]:
#all_data[all_data.episode_fips_id=='128129_45045']

In [21]:
all_data.outage_in_an_hour.mean()

np.float64(0.027549734662457963)

In [17]:
all_data.outage_in_an_hour.mean()

np.float64(0.027549734662457963)

In [22]:
all_data.outage_in_an_hour.value_counts()

outage_in_an_hour
0    904511
1     25625
Name: count, dtype: int64

In [23]:
pd.set_option('display.max_columns', 100)
all_data

,time,T2M,ALLSKY_SFC_SW_DWN,PS,WS50M,ALLSKY_SFC_LW_DWN,WD50M,PRECTOTCORR,GWETPROF,CLRSKY_SFC_SW_DWN,QV10M,RHOA,T10M,TO3,TQV,Z0M,TOA_SW_DWN,RH2M,WS2M,CLRSKY_SFC_LW_DWN,DISPH,episode_fips_id,meteorological_current_datetime_val,day_of_year,hour_of_day,day_of_week,month_of_year,coord0,coord1,coord2,meteorological_nextHour_datetime_val,storm_start,outage_start,outage_end,hours_to_outage,outage_in_an_hour
episode_fips_time_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
100009_55009_2015081411,2015081411,31.06,815.92,99.03,6.17,407.52,293.9,1.69,0.62,838.92,13.15,1.13,30.75,317.29,34.58,0.06,1136.62,48.67,3.49,402.10,0.30,100009_55009,2015-08-14 11:00:00,226,11,4,8,-88.004,44.452,223.21,2015-08-14 12:00:00,2015-08-14 13:30:00,2015-08-14 22:00:00,2015-08-15 05:15:00,11.0,0
100009_55009_2015081412,2015081412,31.48,800.83,99.00,6.42,411.45,294.3,2.84,0.62,836.10,13.31,1.13,31.24,316.79,34.94,0.06,1135.12,47.98,3.62,408.40,0.30,100009_55009,2015-08-14 12:00:00,226,12,4,8,-88.004,44.452,223.21,2015-08-14 13:00:00,2015-08-14 13:30:00,2015-08-14 22:00:00,2015-08-15 05:15:00,10.0,0
100009_55009_2015081413,2015081413,31.57,639.47,98.97,6.70,419.52,296.0,5.70,0.61,783.12,13.60,1.12,31.40,314.92,35.27,0.06,1071.88,48.60,3.74,404.27,0.30,100009_55009,2015-08-14 13:00:00,226,13,4,8,-88.004,44.452,223.21,2015-08-14 14:00:00,2015-08-14 13:30:00,2015-08-14 22:00:00,2015-08-15 05:15:00,9.0,0
100009_55009_2015081414,2015081414,31.37,343.15,98.95,6.92,446.12,298.4,8.90,0.61,680.40,14.00,1.12,31.33,312.90,35.38,0.06,951.17,50.34,3.81,403.17,0.30,100009_55009,2015-08-14 14:00:00,226,14,4,8,-88.004,44.452,223.21,2015-08-14 15:00:00,2015-08-14 13:30:00,2015-08-14 22:00:00,2015-08-15 05:15:00,8.0,0
100009_55009_2015081415,2015081415,30.69,123.62,98.94,7.12,429.02,299.7,10.35,0.61,536.53,14.28,1.13,30.75,311.98,35.07,0.06,781.22,53.10,3.83,402.08,0.30,100009_55009,2015-08-14 15:00:00,226,15,4,8,-88.004,44.452,223.21,2015-08-14 16:00:00,2015-08-14 13:30:00,2015-08-14 22:00:00,2015-08-15 05:15:00,7.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99992_55105_2015091801,2015091801,21.45,0.00,97.79,6.96,397.35,221.4,89.98,0.59,0.00,15.89,1.15,21.68,292.15,47.97,0.08,0.00,96.91,2.66,383.38,0.39,99992_55105,2015-09-18 01:00:00,261,1,4,9,-89.071,42.671,261.74,2015-09-18 02:00:00,2015-09-17 13:25:00,2015-09-18 05:00:00,2015-09-18 07:45:00,4.0,0
99992_55105_2015091802,2015091802,21.43,0.00,97.76,5.75,398.52,242.1,87.26,0.59,0.00,16.06,1.14,21.68,291.00,47.67,0.08,0.00,97.72,1.84,380.92,0.39,99992_55105,2015-09-18 02:00:00,261,2,4,9,-89.071,42.671,261.74,2015-09-18 03:00:00,2015-09-17 13:25:00,2015-09-18 05:00:00,2015-09-18 07:45:00,3.0,0
99992_55105_2015091803,2015091803,21.34,0.00,97.75,5.17,386.92,271.6,72.51,0.60,0.00,16.05,1.14,21.62,290.40,45.89,0.08,0.00,98.08,1.46,379.55,0.39,99992_55105,2015-09-18 03:00:00,261,3,4,9,-89.071,42.671,261.74,2015-09-18 04:00:00,2015-09-17 13:25:00,2015-09-18 05:00:00,2015-09-18 07:45:00,2.0,0


In [24]:
sample_val = all_data[all_data.outage_in_an_hour == 1].sample(1).iloc[0]
print(sample_val.meteorological_current_datetime_val, sample_val.outage_start)
storm_outages_g[storm_outages_g.episode_fips_id == sample_val.episode_fips_id]

2017-07-24 02:00:00 2017-07-24 02:15:00


,episode_fips_id,EPISODE_ID,fips_code_id,episode_description,begin_datetime,end_datetime,storm_duration,storm_caused_outage,outage_index_id,outage_duration,run_start_time_min,run_start_time_max
62105,118086_04013,118086,04013,Thunderstorms with heavy rain developed across...,2017-07-23 18:30:00,2017-07-23 22:45:00,4.25,1.0,04013__0057,0.708333,2017-07-24 02:15:00,2017-07-24 19:15:00


In [ ]:
all_data.drop(
    ['meteorological_nextHour_datetime_val', 
     'storm_start', 'outage_start', 'outage_end',], axis=1
).to_parquet('../data/interim/meteorological_data_with_outages.parquet')